In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_data = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

In [ ]:
train_data

In [ ]:
train_label = train_data['label'].to_numpy()
train_data.drop(columns = ['label'], inplace=True)
train_data = train_data.to_numpy()

In [ ]:
train_data.shape, train_label.shape

In [ ]:
labels, count = np.unique(train_label, return_counts = True)
plt.bar(labels, count, width = 0.7, color = 'green')
plt.xticks(labels)
# plt.pie(count, labels = labels)
plt.show()

In [ ]:
from torch import nn
from torchvision import transforms
augmentation_transform = transforms.Compose([
    transforms.ToTensor(),
    # RandomRotation(degrees=(0, 30),
    transforms.RandomAffine(degrees=(0, 30), translate=(0.15, 0.15)),
    # transforms.GaussianBlur(kernel_size=3),
    # transforms.RandomPerspective(distortion_scale=0.3, p=1.0),
])

In [ ]:
random_value = np.random.randint(len(train_data))
image = train_data[random_value].reshape(28,28)
plt.figure(figsize=(10,5))
plt.subplot(121)
plt.imshow(image, cmap='gray')
plt.title(f'{train_label[random_value]} - real')
plt.axis('off')
plt.subplot(122)
plt.imshow(augmentation_transform(image)[0], cmap = 'gray')
plt.title(f'{train_label[random_value]} - augment')
plt.axis('off')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train_data, train_label, test_size = 0.2)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

In [ ]:
# from sklearn.preprocessing import MinMaxScaler, Normalizer

# X_train_scaler = Normalizer('max').fit_transform(X_train)
# X_val_scaler = Normalizer('max').fit_transform(X_val)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class CustomDataset(Dataset):
    def __init__(self, data, labels, transform = None):
        self.data = data
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        if self.transform:
            image = self.transform(self.data[idx].reshape(28, 28)).to(torch.float32)
        else:
            image = torch.tensor(
                self.data[idx].reshape(1, 28, 28), 
                dtype=torch.float32
            )
        
        label = torch.tensor(
            self.labels[idx], 
            dtype=torch.long
        )
        
        return image / image.max(), label

In [ ]:
X_train_dataset = CustomDataset(X_train, y_train)
X_val_dataset = CustomDataset(X_val, y_val)

In [ ]:
BATCH_SIZE = 32
train_loader = DataLoader(X_train_dataset, batch_size = BATCH_SIZE)
val_loader = DataLoader(X_val_dataset, batch_size = BATCH_SIZE)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# device = 'cuda'
print(f"Using {device} device")

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 16, 3),
            nn.ReLU(),
            nn.BatchNorm2d(16),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 3),
            nn.ReLU(),
            nn.BatchNorm2d(32)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Dropout2d(p=0.5)
        )
        
        self.fc1 = nn.Sequential(
            nn.Linear(64*22*22, 512),
            nn.ReLU(),
            nn.Dropout(p=0.5)
        )
        # self.fc2 = nn.Sequential(
        #     nn.Linear(512, 256),
        #     nn.ReLU(),
        #     nn.Dropout(p=0.5)
        # )
        self.fc3 = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU()
        )
        self.fc4 = nn.Linear(128, 10)


    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        # x = self.fc2(x)
        x = self.fc3(x)
        x = self.fc4(x)
        return nn.functional.log_softmax(x, dim=1)
        

In [ ]:
model = ConvNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss()

In [ ]:
def calculate_accuracy(loader, model):
    correct = 0
    total = 0
    with torch.no_grad():
        for data in loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

In [ ]:
from tqdm import tqdm
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
EPOCHS = 25
tqdm_looper = tqdm(range(EPOCHS), desc="Training")

for epoch in tqdm_looper:
    model.train()
    train_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() / images.size(0)

    train_acc = calculate_accuracy(train_loader, model)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    
    val_loss = 0
    model.eval()
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() / images.size(0) 
            
    val_acc = calculate_accuracy(val_loader, model)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
        
    
    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}')


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(val_accuracies, label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
class TestDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = torch.tensor(
            self.data[idx].reshape(1, 28, 28), 
            dtype=torch.float32
        )
        return image / image.max()

In [ ]:
test_dataset = TestDataset(test_data.to_numpy())

In [ ]:
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False)

In [ ]:
def test(loader, model):
    out = []
    with torch.no_grad():
        for data in loader:
            images = data
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            out.append(predicted)
    return out

In [ ]:
out = test(test_loader, model)

In [ ]:
values = []
for batch in out:
    values.extend(batch.cpu().numpy())

In [ ]:
values = np.array(values)

In [ ]:
submission =  pd.DataFrame({'ImageId': np.arange(1, len(test_data)+1), 'Label':values})

In [ ]:
submission.to_csv('/kaggle/working/submission.csv', index = False)